In [4]:
import os
import time
import subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pennylane as qml
from dataclasses import dataclass
import nbformat

In [ ]:
#  %run -i path_of_Quantum_DReQNN.ipynb

In [ ]:
epsilon_ = [0.06, 0.12, 0.18, 0.24, 0.3]
for index, epsilon in enumerate(epsilon_):
    config_b = QMLConfig()
    config_b.seed = 42
    config_b.n_qubits = 4
    config_b.n_layers = 5
    config_b.n_classes = 4
    # config_b.data_path = "../../PAC/processed_kmnist.pt" # replace with your path
    
    # Result save directory (only folder specified, filename auto-generated)
    config_b.save_dir = "./results"
    
    # Set to -1 for automatic GPU selection, otherwise manually specify 0, 1, ...
    config_b.device_id = -1
    
    # --- Training settings ---
    config_b.batch_size = 64
    config_b.learning_rate = 0.005
    config_b.num_epochs = 100
    
    # --- Data sampling ---
    config_b.n_train_limit = 2000
    config_b.n_test_limit = 400
    
    # --- Adversarial attack parameters ---
    config_b.train_epsilon = epsilon
    config_b.train_alpha = 0.01 * (index + 1)
    config_b.train_iter = 3
    
    config_b.test_epsilon = epsilon + 0.02
    config_b.test_alpha = 0.01 * (index + 1)
    config_b.test_iter = 10
    
    # 3. Run
    print("\n=== Starting strong adversarial experiment ===")
    trainer_b = QuantumAdversarialTrainer(config_b)
    model_b = trainer_b.run()
    del trainer_b
    torch.cuda.empty_cache()